In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

In [ ]:
from astropy.io import fits
import os
import numpy as np
import jax.numpy as jnp

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions

import matplotlib.pyplot as plt

In [ ]:
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.mass.shear import Shear
from gigalens.jax.profiles.mass.nfw import NFW_ELLIPSE

from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.profiles.light.shapelets import Shapelets

from gigalens.jax.cosmo import wCDM_Cosmo

from gigalens.jax.scene_prob_model import Dataset, ProbModel
from gigalens.simulator import SimulatorConfig
from gigalens.jax.inference import ModellingSequence

In [ ]:
# def prior():
#     lhalo = Prior(mass.nfw_mod.NFW_ELLIPSE(),
#                    dict(
#                        Rs = tfd.Uniform(20,100),
#                        alpha_Rs = tfd.Uniform(10,40),
#                        e1 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
#                        e2 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
#                        center_x = tfd.Normal(5.344, 0.05),
#                        center_y = tfd.Normal(3.805, 0.05)
#                    )
#     )

#     ld = Prior(mass.epl.EPL(),
#                     dict(
#                         theta_E = tfd.TruncatedNormal(1.6730331, 0.1, 1, 2.5),
#                         gamma = tfd.Uniform(1,3),
#                         e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                         e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                         center_x = tfd.Normal(11.80977389, 0.1),
#                         center_y = tfd.Normal(23.0283886, 0.1)
#                     )

#     )

#     le = Prior(mass.epl.EPL(),
#                        dict(
#                            theta_E = tfd.TruncatedNormal(2.4, 0.1, 1, 3),
#                            gamma = tfd.TruncatedNormal(2.2, 0.5, 1, 3),
#                            e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                            e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                            center_x = tfd.Normal(-22.1, 0.1),
#                            center_y = tfd.Normal(-24.7, 0.1)
#                        )
#     )

#     lf = Prior(mass.epl.EPL(),
#                        dict(
#                            center_x = tfd.Normal(-15.10088063, 0.1),
#                            center_y = tfd.Normal(-4.66657821, 0.1),
#                            e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                            e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                            theta_E = tfd.TruncatedNormal(0.8151327, 0.05, 0.2, 1.5),
#                            gamma = tfd.TruncatedNormal(2.2266, 0.5, 1, 3)
#                        )
#     )

#     shear_model = Prior(mass.shear.Shear(),
#                         dict(
#                             gamma1 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
#                             gamma2 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
#                         ))
#     source_model = [
#         # Source 1
#         Prior(
#             light.sersic_shapelets.SersicShapelets(9, use_lstsq=True, interpolate=False),
#             dict(
#                 z_source = 0.962,
#                 center_x = tfd.Normal(7.67187389, 2),
#                 center_y = tfd.Normal(3.31911655, 2),
#                 e1 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
#                 e2 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
#                 n_sersic = tfd.Uniform(0.1,15),
#                 R_sersic = tfd.LogNormal(jnp.log(0.1), 0.15),
#                 beta = tfd.LogNormal(jnp.log(0.1), 0.15),
#             )
#         ),

#         # Source 3
#         Prior(
#             light.shapelets.Shapelets(12, use_lstsq=True, interpolate=False),
#             dict(
#                 z_source = 1.166,
#                 center_x = tfd.Normal(5., 1),
#                 center_y = tfd.Normal(5., 1),
#                 beta = tfd.LogNormal(jnp.log(0.4), 0.15)
#             )
#         ),
#         # Source 4 and 5
#         Prior(
#             light.combined_profile.CombinedProfile(
#                 profiles=[
#                     light.shapelets.Shapelets(12, use_lstsq=True, interpolate=False),
#                     light.shapelets.Shapelets(6, use_lstsq=True, interpolate=False),
#                 ],
#                 shared_params=[],
#                 use_lstsq=True,
#             ),
#             dict(
#                 z_source = 1.432,
#                 center_x_0 = tfd.Normal(3.7, 1),
#                 center_y_0 = tfd.Normal(3.2, 1),
#                 beta_0 = tfd.LogNormal(jnp.log(0.4), 0.15),
#                 center_x_1 = tfd.Normal(3.0, 1),
#                 center_y_1 = tfd.Normal(0., 1),
#                 beta_1 = tfd.LogNormal(jnp.log(0.1), 0.15),
#             )
#         ),
#         # Source 9
#         Prior(
#             light.sersic.SersicEllipse(use_lstsq=True),
#             dict(
#                 z_source = 1.506,
#                 center_x = tfd.Normal(-10, 1),
#                 center_y = tfd.Normal(-16, 1),
#                 n_sersic = tfd.Uniform(0.1,10),
#                 R_sersic = tfd.LogNormal(jnp.log(0.4), 0.15),
#                 e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#                 e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#             )
#         ),
#     ]
#     cosmo_model = wCDM_Cosmo(0.49, 1.432)
#     cosmo_prior = Prior(
#             cosmo_model,  # you need to set the redshifts for the cosmology to work, theta_E is relative to z_source_ref
#         dict(
#             H0=70.,
#             Om0=0.3,
#             w0=-1,
#             # wa=0.0,
#             k=0.0,
#         )
#     )
#     prior, phys_model = make_prior_and_model(
#         lenses=[lhalo, ld, le, lf, shear_model],
#         sources=[*source_model],
#         cosmo=cosmo_prior
#     )
#     return prior, phys_model

In [ ]:
#* MASS PRIORS
NFW0 = Component(NFW_ELLIPSE(), dict(
   Rs = tfd.Uniform(20,100),
   alpha_Rs = tfd.Uniform(10,40),
   e1 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
   e2 = tfd.TruncatedNormal(0, 0.05, -0.2, 0.2),
   center_x = tfd.Normal(5.344, 0.05),
   center_y = tfd.Normal(3.805, 0.05)
))

# EPL_Le = Component(EPL(18), dict(
#    theta_E = tfd.TruncatedNormal(2.4, 0.1, 1, 3),
#    gamma = tfd.TruncatedNormal(2.2, 0.5, 1, 3),
#    e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#    e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#    center_x = tfd.Normal(-22.1, 0.1),
#    center_y = tfd.Normal(-24.7, 0.1)
# ))

#* Comes along with source 3
# EPL_Ld = dict(
#     theta_E = tfd.TruncatedNormal(1.6730331, 0.1, 1, 2.5),
#     gamma = tfd.Uniform(1,3),
#     e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#     e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#     center_x = tfd.Normal(11.80977389, 0.1),
#     center_y = tfd.Normal(23.0283886, 0.1)
# )

# EPL_Lf = Component(EPL(18), dict(
#    center_x = tfd.Normal(-15.10088063, 0.1),
#    center_y = tfd.Normal(-4.66657821, 0.1),
#    e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#    e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#    theta_E = tfd.TruncatedNormal(0.8151327, 0.05, 0.2, 1.5),
#    gamma = tfd.TruncatedNormal(2.2266, 0.5, 1, 3)
# ))

shear = Component(Shear(), dict(
    gamma1 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
    gamma2 = tfd.TruncatedNormal(0., 0.1, -0.3, 0.3),
))

#* LIGHT PRIORS

src4 = Component(Shapelets(n_max=8, use_lstsq=True), dict( #default n_max=12
    center_x = tfd.Normal(3.7, 1),
    center_y = tfd.Normal(3.2, 1),
    beta = tfd.LogNormal(jnp.log(0.4), 0.15),
))

src5 = Component(Shapelets(n_max=6, use_lstsq=True), dict(
    center_x = tfd.Normal(3.0, 1),
    center_y = tfd.Normal(0., 1),
    beta = tfd.LogNormal(jnp.log(0.1), 0.15),
))

# src9 = Component(SersicEllipse(use_lstsq=True), dict( #default n_max=12
#     center_x = tfd.Normal(-10, 1),
#     center_y = tfd.Normal(-16, 1),
#     n_sersic = tfd.Uniform(0.1,10),
#     R_sersic = tfd.LogNormal(jnp.log(0.4), 0.15),
#     e1 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
#     e2 = tfd.TruncatedNormal(0, 0.1, -0.3, 0.3),
# ))

In [ ]:
#* MASS PARAMETERS
# NFW0 = {
#     'Rs': 37.439292907714844,
#     'alpha_Rs': 16.946298599243164,
#     'center_x': 5.350801467895508,
#     'center_y': 3.905395984649658,
#     'e1': -0.0517449676990509,
#     'e2': 0.030034080147743225
# }

# EPL_Le = {
#     'center_x': -22.150279998779297,
#     'center_y': -24.73190689086914,
#     'e1': 0.05808083713054657,
#     'e2': 0.29447436332702637,
#     'gamma': 2.199589967727661,
#     'theta_E': 2.3904967308044434
# }

# EPL_Lf = {
#     'center_x': -14.754530906677246,
#     'center_y': -4.7380170822143555,
#     'e1': 0.2831626236438751,
#     'e2': -0.20151233673095703,
#     'gamma': 2.59816312789917,
#     'theta_E': 0.9524290561676025
# }

# shear = {'gamma1': 0.027523696422576904, 'gamma2': -0.012137502431869507}


#* LIGHT PARAMETERS

# src3 = Component(Shapelets(n_max=12, use_lstsq=True), {
#     'beta': tfd.LogNormal(jnp.log(0.5), 0.2),
#     'center_x': tfd.Normal(5, 2),
#     'center_y': tfd.Normal(5, 2),
# })

# src4 = Component(Shapelets(n_max=12, use_lstsq=True), {
#     'beta': 0.24978873133659363,
#     'center_x': 3.2731070518493652,
#     'center_y': 3.130897045135498,
# })

# src5 = Component(Shapelets(n_max=6, use_lstsq=True), {
#     'beta': 0.20333564281463623,
#     'center_x': 3.623133897781372,
#     'center_y': -0.2059929519891739,
# })
    
# src9 = Component(SersicEllipse(use_lstsq=True),{
#     'R_sersic': 0.4022407829761505,
#     'center_x': -10.395180702209473,
#     'center_y': -16.07061767578125,
#     'e1': 0.279153972864151,
#     'e2': 0.29133689403533936,
#     'n_sersic': 2.795565128326416
# })

In [ ]:
#* All the physical stuff
z1=0.962
z3=1.166
z4_5=1.432
z9=1.506
z12_13=3.086
z8=3.549
z11=4.090

z_lens = 0.49


cosmo = Component(wCDM_Cosmo(z_lens=z_lens, z_source_ref=z4_5), dict(H0=70.0, Om0=0.3, k=0.0, w0=-1.0))

model = LensModel([
    Plane(redshift=z_lens, mass=[NFW0, shear]),       # the one deflector #EPL_Le, EPL_Lf,
    # Plane(redshift=z1, light=[src1]),    # nearer source plane
    # Plane(redshift=z3, light=[src3]),
    Plane(redshift=z4_5, light=[src4, src5]),
    # Plane(redshift=z9, light=[src9]),
    # Plane(redshift=z12, light=[src12, src13]),
    # Plane(redshift=z8, light=[src8]),
    # Plane(redshift=z11, light=[src11]),
], cosmo=cosmo)

In [ ]:
def dataset_from_dir(path, ext):

    img_path = os.path.join(path, f"source{ext}.fits")
    with fits.open(img_path) as hdul:
        observed_image = jnp.array(hdul['DATA'].data.astype("float64"))

        error_map = jnp.array(np.sqrt(hdul['STAT'].data.astype("float64")))
        # background_rms = hdul['DATA'].header['BKG_RMS']
        # exp_time = hdul['PRIMARY'].header['EXPTIME']
            # if centroids is None: (self.centroids_x, self.centroids_y) = Table(hdul['CENTROIDS'].data)['centroid'].data.T
            # if centroids_error is None: self.centroids_error = Table(hdul['CENTROIDS'].data)['sky_covariance'].data
        psf = hdul['PSF'].data.astype(jnp.float64)
        mask = hdul['MASK'].data.astype(jnp.bool)
        # hot_pix = jnp.load(os.path.join(path, f"hot_pix.npy"))

    # mask = jnp.logical_and(mask, hot_pix)

    return observed_image, error_map, psf, mask

path= "newnewcutouts/"
def ds(ext, sees):
    observed_image, error_map, psf, mask = dataset_from_dir(path, ext)

    cfg = SimulatorConfig(delta_pix=0.2, num_pix=300, supersample=1, kernel=psf, likelihood_precision="float64", conv_precision="float32")
    dset = Dataset(observed_image, cfg, error_map=error_map, mask=mask, sees=sees)
    return dset
    

# d1 = ds("1", sees=[src1])
# d3 = ds("3", sees=[src3])
d4_5 = ds("4-5", sees=[src4, src5])
# d9 = ds("9", sees=[src9])
# d12_13 = ds("12-13", sees=[src12_13])
# d8 = ds("8", sees=[src8])
# d11 = ds("11", sees=[src11])

prob_model = ProbModel(model, [d4_5], mode="lstsq")

# sim = SceneSimulator(model, cfg)

In [ ]:
from gigalens_research.inference_utils import (
    InferenceContext, Pipeline, MAPStage, BridgeStage, MCLMCStage#, SVIStage, HMCStage,
)
model_seq = ModellingSequence(prob_model)
ctx = InferenceContext.from_modelling_sequence(model_seq)
pipeline = Pipeline(ctx, seed=42)
pipeline.add(MAPStage(num_steps=500, n_samples=64))


def make_diag_qz(z_best):
    return tfd.MultivariateNormalDiag(
        loc=jnp.asarray(z_best),
        scale_diag=jnp.full(z_best.shape[-1], 1e-3),
    )

pipeline.add(BridgeStage(
    name="diag_qz",
    version="v1",           # bump to 'v2' if you change the scale or logic
    requires=("z_best",),
    produces=("qz",),
    fn=make_diag_qz,
))

pipeline.add(MCLMCStage(n_chains=8, num_burnin_steps=10000, num_results=10000, debug=True, progress_bar=True, regularize_mass_matrix=True))

artifacts = pipeline.run(out_dir="messy_tests/minimal_case", resume=True)

In [ ]:
from gigalens_research.plotting import PosteriorReport, PipelineReport

pipeline_report = PipelineReport(pipeline)
fig = pipeline_report.diagnostics("mclmc", chain=3)
fig.show()
fig = pipeline_report.diagnostics_surrogate_corner("mclmc")
fig.show()

In [ ]:

# import matplotlib.pyplot as plt
# import matplotlib.colors as colors
# im = sim9.lstsq_simulate(model.to_params({}), d9.image, err_map=d9.error_map, mask=d9.mask)
# plt.imshow(im, norm=colors.PowerNorm(gamma=0.5, vmin=0), origin="lower")
# plt.colorbar()
# plt.show()

In [ ]:
# im = sim4_5.lstsq_simulate(model.to_params({}), d4_5.image, err_map=d4_5.error_map, mask=d4_5.mask)
# plt.imshow(im, norm=colors.PowerNorm(gamma=0.5, vmin=0), origin="lower")
# plt.colorbar()
# plt.show()

In [ ]:
# model.to_params({})

In [ ]:
# mask = d9.mask
# masked_img = jnp.where(mask, d9.image, 0)
# plt.imshow(masked_img,norm=colors.PowerNorm(gamma=0.5, vmin=0))
# plt.colorbar()
# plt.show()

In [ ]:
# plt.imshow(d4_5.mask)
# plt.show()

In [ ]:
# jnp.sum(d4_5.mask)
# np.unique(d4_5.mask)

In [ ]:
# from gigalens.jax.scene_simulator import SceneSimulator
# from gigalens.simulator import SimulatorConfig

